In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE

In [2]:
# =====================================================================
# 1. SIMULATING HIGHLY IMBALANCED FRAUD DATA
# =====================================================================
np.random.seed(42)
n_samples = 5000
n_features = 5

# Generate features (e.g., transaction amount, distance, velocity)
X_raw = np.random.normal(0, 1, size=(n_samples, n_features))
# Force class imbalance: 98% Legitimate (0), 2% Fraudulent (1)
y_raw = np.random.choice([0, 1], size=n_samples, p=[0.98, 0.02])

feature_names = [f"Feature_{i}" for i in range(n_features)]
df = pd.DataFrame(X_raw, columns=feature_names)
df["Is_Fraud"] = y_raw

print(f"Dataset Shape: {df.shape}")
print(f"Class Distribution:\n{df['Is_Fraud'].value_counts(normalize=True)}\n")

Dataset Shape: (5000, 6)
Class Distribution:
Is_Fraud
0    0.9792
1    0.0208
Name: proportion, dtype: float64



In [3]:
# =====================================================================
# 2. RIGOROUS TRAIN-TEST SPLIT (Avoiding Data Leakage)
# =====================================================================
X = df.drop(columns=["Is_Fraud"])
y = df["Is_Fraud"]

# Stratify=y ensures both sets have the exact same ratio of fraud
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
# =====================================================================
# 3. BALANCING THE SCALES VIA SMOTE (Applied ONLY to Training Set)
# =====================================================================
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f"Training Class Distribution After SMOTE:\n{pd.Series(y_train_resampled).value_counts()}\n")

Training Class Distribution After SMOTE:
Is_Fraud
0    3917
1    3917
Name: count, dtype: int64



In [5]:
# =====================================================================
# 4. MODEL TRAINING & COMPARISON
# =====================================================================

# Model A: Baseline Logistic Regression
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_resampled, y_train_resampled)
lr_preds = lr_model.predict(X_test_scaled)
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]

# Model B: Advanced Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_resampled, y_train_resampled)
rf_preds = rf_model.predict(X_test_scaled)
rf_probs = rf_model.predict_proba(X_test_scaled)[:, 1]

In [6]:
# =====================================================================
# 5. STRICT METRIC EVALUATION (Discarding Pure Accuracy)
# =====================================================================
print("=================== LOGISTIC REGRESSION RESULTS ===================")
print(classification_report(y_test, lr_preds))
print(f"ROC-AUC Score: {roc_auc_score(y_test, lr_probs):.4f}\n")

print("=================== RANDOM FOREST RESULTS ===================")
print(classification_report(y_test, rf_preds))
print(f"ROC-AUC Score: {roc_auc_score(y_test, rf_probs):.4f}\n")

print("Confusion Matrix (Random Forest):")
print(confusion_matrix(y_test, rf_preds))

=================== LOGISTIC REGRESSION RESULTS ===================
              precision    recall  f1-score   support

           0       0.98      0.53      0.69       979
           1       0.03      0.62      0.05        21

    accuracy                           0.54      1000
   macro avg       0.51      0.58      0.37      1000
weighted avg       0.96      0.54      0.68      1000

ROC-AUC Score: 0.5488

=================== RANDOM FOREST RESULTS ===================
              precision    recall  f1-score   support

           0       0.98      0.96      0.97       979
           1       0.05      0.10      0.06        21

    accuracy                           0.94      1000
   macro avg       0.51      0.53      0.52      1000
weighted avg       0.96      0.94      0.95      1000

ROC-AUC Score: 0.4486

Confusion Matrix (Random Forest):
[[939  40]
 [ 19   2]]
